# Project 3: SQL Data Analysis 🗄️
**DecodeLabs Industrial Training Kit — Batch 2026**

**Prepared by:** Ayesha Sarwar

**Goal:** Use SQL queries to extract business insights from the e-commerce dataset by writing SELECT queries with WHERE, GROUP BY, ORDER BY, and aggregations (COUNT, SUM, AVG).

**Dataset:** `cleaned_dataset.csv` — 1,200 order records, 14 columns.

## 1. Setup — Load Data into SQL Database
We load the CSV into an in-memory SQLite database so we can run real SQL queries on it.

In [1]:
import pandas as pd
import sqlite3

# Load cleaned dataset
df = pd.read_csv('cleaned_dataset.csv')

# Create in-memory SQLite database
conn = sqlite3.connect(':memory:')

# Load dataframe into SQL table called 'orders'
df.to_sql('orders', conn, index=False, if_exists='replace')

print("Database ready. Table 'orders' loaded with", len(df), "rows.")

Database ready. Table 'orders' loaded with 1200 rows.


## 2. Query 1 — SELECT: View First 5 Orders
Basic SELECT query to preview the data.

In [2]:
query1 = "SELECT * FROM orders LIMIT 5"
pd.read_sql_query(query1, conn)

,OrderID,Date,CustomerID,Product,Quantity,UnitPrice,ShippingAddress,PaymentMethod,OrderStatus,TrackingNumber,ItemsInCart,CouponCode,ReferralSource,TotalPrice
0,ORD200000,2023-01-04,C72649,Monitor,5,570.62,928 Main St,Debit Card,Shipped,TRK37947903,7,SAVE10,Instagram,2853.10
1,ORD200001,2024-08-23,C75739,Phone,2,151.35,823 Main St,Online,Shipped,TRK91186779,3,SAVE10,Referral,302.70
2,ORD200002,2024-02-27,C81728,Tablet,5,550.68,512 Main St,Credit Card,Cancelled,TRK42903982,8,FREESHIP,Email,2753.40
3,ORD200003,2023-10-15,C33540,Chair,1,273.19,275 Main St,Debit Card,Returned,TRK62788070,5,SAVE10,Facebook,273.19
4,ORD200004,2025-05-08,C81840,Printer,4,626.01,668 Main St,Online,Delivered,TRK29241424,8,SAVE10,Email,2504.04


## 3. Query 2 — WHERE: Filter Delivered Orders
Use WHERE to isolate only delivered orders.

In [3]:
query2 = """
SELECT OrderID, Product, TotalPrice, OrderStatus
FROM orders
WHERE OrderStatus = 'Delivered'
LIMIT 10
"""
pd.read_sql_query(query2, conn)

,OrderID,Product,TotalPrice,OrderStatus
0,ORD200004,Printer,2504.04,Delivered
1,ORD200015,Printer,473.96,Delivered
2,ORD200021,Monitor,1371.80,Delivered
3,ORD200024,Laptop,461.90,Delivered
4,ORD200025,Monitor,399.98,Delivered
5,ORD200026,Laptop,297.75,Delivered
6,ORD200029,Desk,1270.59,Delivered
7,ORD200032,Tablet,2683.60,Delivered
8,ORD200038,Laptop,245.89,Delivered
9,ORD200039,Printer,769.38,Delivered


## 4. Query 3 — WHERE + ORDER BY: High Value Orders
Find orders above 2000 in TotalPrice, sorted highest first.

In [4]:
query3 = """
SELECT OrderID, Product, Quantity, UnitPrice, TotalPrice
FROM orders
WHERE TotalPrice > 2000
ORDER BY TotalPrice DESC
LIMIT 10
"""
pd.read_sql_query(query3, conn)

,OrderID,Product,Quantity,UnitPrice,TotalPrice
0,ORD200789,Tablet,5,691.28,3456.40
1,ORD201122,Monitor,5,678.19,3390.95
2,ORD200632,Laptop,5,678.16,3390.80
3,ORD200469,Chair,5,676.98,3384.90
4,ORD200328,Tablet,5,674.04,3370.20
5,ORD200107,Printer,5,670.75,3353.75
6,ORD200326,Laptop,5,670.48,3352.40
7,ORD201065,Printer,5,666.80,3334.00
8,ORD201031,Phone,5,664.51,3322.55
9,ORD200463,Laptop,5,662.78,3313.90


## 5. Query 4 — GROUP BY + COUNT: Orders Per Product
Count how many orders each product has.

In [5]:
query4 = """
SELECT Product, COUNT(*) AS Total_Orders
FROM orders
GROUP BY Product
ORDER BY Total_Orders DESC
"""
pd.read_sql_query(query4, conn)

,Product,Total_Orders
0,Printer,181
1,Tablet,179
2,Chair,178
3,Laptop,173
4,Desk,170
5,Monitor,163
6,Phone,156


## 6. Query 5 — GROUP BY + SUM: Total Revenue Per Product
Calculate total revenue generated by each product.

In [6]:
query5 = """
SELECT Product, 
       SUM(TotalPrice) AS Total_Revenue,
       ROUND(AVG(TotalPrice), 2) AS Avg_Order_Value
FROM orders
GROUP BY Product
ORDER BY Total_Revenue DESC
"""
pd.read_sql_query(query5, conn)

,Product,Total_Revenue,Avg_Order_Value
0,Chair,195620.11,1098.99
1,Printer,195612.61,1080.73
2,Laptop,192126.56,1110.56
3,Tablet,186568.95,1042.28
4,Monitor,175651.41,1077.62
5,Desk,167459.93,985.06
6,Phone,151722.39,972.58


## 7. Query 6 — GROUP BY + COUNT: Orders Per Payment Method
Which payment method is most popular?

In [7]:
query6 = """
SELECT PaymentMethod, COUNT(*) AS Total_Orders,
       ROUND(SUM(TotalPrice), 2) AS Total_Revenue
FROM orders
GROUP BY PaymentMethod
ORDER BY Total_Orders DESC
"""
pd.read_sql_query(query6, conn)

,PaymentMethod,Total_Orders,Total_Revenue
0,Online,258,262442.94
1,Cash,246,259786.29
2,Credit Card,234,263847.63
3,Debit Card,232,232361.18
4,Gift Card,230,246323.92


## 8. Query 7 — GROUP BY + HAVING: Products With More Than 150 Orders
HAVING filters after grouping — unlike WHERE which filters before.

In [8]:
query7 = """
SELECT Product, COUNT(*) AS Total_Orders
FROM orders
GROUP BY Product
HAVING Total_Orders > 150
ORDER BY Total_Orders DESC
"""
pd.read_sql_query(query7, conn)

,Product,Total_Orders
0,Printer,181
1,Tablet,179
2,Chair,178
3,Laptop,173
4,Desk,170
5,Monitor,163
6,Phone,156
